# 文字探勘與行銷應用 — 進階爬蟲實戰
**工具：Firecrawl v4 × BeautifulSoup × jieba 中文斷詞 × Google Sheets**

> ✅ 本 notebook 完全免費，不需要任何 AI API 金鑰

## 第 0 步｜安裝套件

In [ ]:
!pip install firecrawl-py jieba gspread google-auth beautifulsoup4 pandas -q

## 第 1 步｜設定 API 金鑰（只需要 Firecrawl 一個）

In [ ]:
FIRECRAWL_API_KEY = "fc-xxxxxxxxxxxxxxxxxxxxxxxx"   # firecrawl.dev 取得（免費 500 次）

## 第 2 步｜載入所有套件

In [ ]:
from firecrawl.v1 import V1FirecrawlApp
from bs4 import BeautifulSoup
from collections import Counter
import jieba
import jieba.analyse
import pandas as pd, re, time
from datetime import datetime

app = V1FirecrawlApp(api_key=FIRECRAWL_API_KEY)

print("✅  所有套件載入完畢")

---
## UNIT 1｜動態網頁爬取基礎

### 1-A｜靜態 vs 動態網頁對比

In [ ]:
# 靜態網頁
result_static = app.scrape_url(
    url="https://www.books.com.tw/products/0010966295",
    formats=["markdown"]
)
print("=== 靜態網頁（前 300 字）===")
print((result_static.markdown or "（無內容）")[:300])

In [ ]:
# 動態網頁：wait_for 等待 JS 渲染（單位毫秒）
# 巴哈姆特論壇文章列表需要 JS 渲染才能取得完整貼文
result_dynamic = app.scrape_url(
    url="https://forum.gamer.com.tw/B.php?bsn=76999",
    formats=["markdown"],
    wait_for=3000
)
print("=== 動態網頁（前 300 字）===")
print((result_dynamic.markdown or "（無內容）")[:300])

### 1-B｜Firecrawl 三種模式：scrape / crawl / map

In [ ]:
scrape_result = app.scrape_url(
    url="https://forum.gamer.com.tw/B.php?bsn=76999",
    formats=["markdown", "html"],
    wait_for=3000
)
print("markdown 字數：", len(scrape_result.markdown or ""))
print("html 字數：    ", len(scrape_result.html or ""))

In [ ]:
# map：取得論壇所有頁面 URL（選用）
# map_result = app.map_url(url="https://forum.gamer.com.tw", limit=20)
# print(map_result.links[:5])
print("（map 已暫時註解，避免消耗 API 配額）")

---
## UNIT 2｜瀑布流（Infinite Scroll）爬取實戰

### 2-A｜Actions 模擬向下捲動

In [ ]:
result_scroll = app.scrape_url(
    url="https://forum.gamer.com.tw/B.php?bsn=76999",
    formats=["markdown", "html"],
    wait_for=3000,
    actions=[
        {"type": "scroll", "direction": "down", "coordinate": [0, 3000]},
        {"type": "wait",   "milliseconds": 1500},
        {"type": "scroll", "direction": "down", "coordinate": [0, 3000]},
        {"type": "wait",   "milliseconds": 1500},
        {"type": "scroll", "direction": "down", "coordinate": [0, 3000]},
        {"type": "wait",   "milliseconds": 2000},
    ]
)

markdown_content = result_scroll.markdown or ""
html_content     = result_scroll.html     or ""

print(f"markdown 字數：{len(markdown_content)}")
print(f"html 字數：    {len(html_content)}")
print("\n--- 前 500 字 ---")
print(markdown_content[:500])

### 2-B｜BeautifulSoup 解析標題與連結

In [ ]:
soup = BeautifulSoup(html_content, "html.parser")
data, seen = [], set()

# 巴哈論壇貼文連結格式：/C.php?bsn=XXXXX&snA=XXXXX
for tag in soup.find_all("a", href=True):
    href  = tag.get("href", "")
    title = tag.get_text(strip=True)
    # 篩選：包含 /C.php 的貼文連結，標題長度合理
    if "/C.php" in href and "bsn=76999" in href and len(title) > 3 and href not in seen:
        seen.add(href)
        full = (GAMER_BASE + href) if href.startswith("/") else href
        data.append({"title": title, "url": full})

GAMER_BASE = "https://forum.gamer.com.tw"
df_posts = pd.DataFrame(data)
print(f"解析出 {len(df_posts)} 筆貼文")
print(df_posts.head(10).to_string(index=False))

### 2-C｜迴圈多輪爬取，累積更多貼文

In [ ]:
GAMER_BASE    = "https://forum.gamer.com.tw"
all_data, scroll_rounds = [], 3

for round_num in range(1, scroll_rounds + 1):
    print(f"第 {round_num} 輪爬取中...")

    r = app.scrape_url(
        url="https://forum.gamer.com.tw/B.php?bsn=76999",
        formats=["html"],
        wait_for=3000,
        actions=[
            {"type": "scroll", "direction": "down",
             "coordinate": [0, round_num * 3000]},
            {"type": "wait", "milliseconds": 2000},
        ]
    )

    soup_r = BeautifulSoup(r.html or "", "html.parser")
    for tag in soup_r.find_all("a", href=True):
        href  = tag.get("href", "")
        title = tag.get_text(strip=True)
        if "/C.php" in href and "bsn=76999" in href and len(title) > 3:
            full = (GAMER_BASE + href) if href.startswith("/") else href
            all_data.append({"title": title, "url": full, "round": round_num})

    time.sleep(2)

df_all = pd.DataFrame(all_data).drop_duplicates(subset=["url"]).reset_index(drop=True)
print(f"\n累積去重後共 {len(df_all)} 筆")
print(df_all.head(10).to_string(index=False))

### 2-D｜jieba 中文斷詞 — 不需要 AI API 的文字分析

In [ ]:
# ── Step 1：TF-IDF 關鍵字萃取 ────────────────────────────────────────────────
keywords = jieba.analyse.extract_tags(
    markdown_content,
    topK=20,
    withWeight=True
)

print("=== 🔑 熱門關鍵字 TOP 20 ===")
for word, weight in keywords:
    bar = "█" * int(weight * 200)
    print(f"  {word:<10} {bar}  ({weight:.4f})")

In [ ]:
# ── Step 2：貼文標題高頻詞統計 ───────────────────────────────────────────────
all_words = []
for title in df_posts["title"]:
    words = [w for w in jieba.cut(title) if len(w) >= 2]
    all_words.extend(words)

word_freq = Counter(all_words).most_common(15)

print("=== 📊 貼文標題高頻詞 TOP 15 ===")
for word, count in word_freq:
    bar = "█" * count
    print(f"  {word:<8} {bar}  ({count} 次)")

In [ ]:
# ── Step 3：規則式情緒分類（關鍵字字典法，零成本）────────────────────────────
positive_words = [
    "推薦", "好用", "喜歡", "讚", "開心", "感謝", "優質",
    "實用", "厲害", "棒", "美", "愛", "順", "方便", "划算",
    "滿意", "值得", "期待", "有趣", "驚喜", "溫馨", "感動"
]
negative_words = [
    "爛", "差", "討厭", "失望", "崩潰", "難用", "詐騙",
    "垃圾", "生氣", "傻眼", "雷", "坑", "貴", "慘", "糟",
    "抱怨", "踩雷", "後悔", "浪費", "問題", "無言", "無奈"
]

def rule_sentiment(text: str) -> str:
    pos = sum(1 for w in positive_words if w in text)
    neg = sum(1 for w in negative_words if w in text)
    if pos > neg:   return "正向"
    elif neg > pos: return "負向"
    else:           return "中立"

df_posts["sentiment"] = df_posts["title"].apply(rule_sentiment)

print("=== 💬 情緒分布 ===")
print(df_posts["sentiment"].value_counts())
print()
print("=== ✅ 正向貼文範例（前 5 筆）===")
print(df_posts[df_posts["sentiment"]=="正向"]["title"].head(5).to_string(index=False))
print()
print("=== ❌ 負向貼文範例（前 5 筆）===")
print(df_posts[df_posts["sentiment"]=="負向"]["title"].head(5).to_string(index=False))

---
## UNIT 3｜登入機制：Cookie / Header / Actions

### 3-A｜帶入 Cookie 與 User-Agent

In [ ]:
# ── Unit 3-A：帶入 Cookie 與 User-Agent — 以巴哈姆特為示範目標 ────────────────
#
# 示範情境：巴哈姆特特定版本需要登入才能看完整內容
# 如何取得 Cookie？
#   1. 瀏覽器登入 https://www.gamer.com.tw
#   2. F12 → Application → Cookies → gamer.com.tw
#   3. 複製 _ga、ck 等 Cookie 的值

TARGET_URL = "https://forum.gamer.com.tw/B.php?bsn=76999"

# ── Step 1：不帶 Cookie（對照組）────────────────────────────────────────────
result_no_cookie = app.scrape_url(
    url=TARGET_URL,
    formats=["markdown"],
    wait_for=3000,
)
print("=== ❌ 不帶 Cookie（前 300 字）===")
print((result_no_cookie.markdown or "（無內容）")[:300])
print()

# ── Step 2：帶 User-Agent Header（模擬真實瀏覽器）────────────────────────────
result_with_ua = app.scrape_url(
    url=TARGET_URL,
    formats=["markdown"],
    wait_for=3000,
    headers={
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Referer": "https://www.gamer.com.tw/",
        "Accept-Language": "zh-TW,zh;q=0.9,en-US;q=0.8",
    },
    actions=[
        {"type": "scroll", "direction": "down", "coordinate": [0, 2000]},
        {"type": "wait",   "milliseconds": 1500},
    ]
)
print("=== ✅ 帶 User-Agent Header（前 500 字）===")
print((result_with_ua.markdown or "（無內容）")[:500])
print()

# ── Step 3：帶完整 Cookie（需先登入取得，選填）───────────────────────────────
# MY_COOKIE = "YOUR_GAMER_COOKIE_HERE"  # ← 替換為從 F12 複製的 Cookie 值

# result_with_cookie = app.scrape_url(
#     url=TARGET_URL,
#     formats=["markdown"],
#     wait_for=3000,
#     headers={
#         "User-Agent": (
#             "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#             "AppleWebKit/537.36 (KHTML, like Gecko) "
#             "Chrome/124.0.0.0 Safari/537.36"
#         ),
#         "Cookie": MY_COOKIE,
#         "Referer": "https://www.gamer.com.tw/",
#     },
# )
# print("=== 🔑 帶 Cookie 登入爬取（前 500 字）===")
# print((result_with_cookie.markdown or "（無內容）")[:500])

### 3-B｜Actions 模擬填表登入

In [ ]:
# ⚠️ 請使用自己的巴哈姆特帳號，勿用於未授權帳號
#
# 如何找到巴哈登入頁面的 CSS Selector？
#   F12 → 點選帳號輸入框 → 右鍵 → Copy → Copy selector

MY_USERNAME = "your_gamer_account"   # ← 替換為你的巴哈帳號
MY_PASSWORD = "your_gamer_password"  # ← 替換為你的巴哈密碼

# result_login = app.scrape_url(
#     url="https://user.gamer.com.tw/login.php",
#     formats=["markdown"],
#     wait_for=5000,
#     actions=[
#         # Step 1：點擊帳號輸入框
#         {"type": "click",  "selector": "input[name='userid']"},
#         # Step 2：輸入帳號
#         {"type": "write",  "text": MY_USERNAME},
#         # Step 3：點擊密碼輸入框
#         {"type": "click",  "selector": "input[name='password']"},
#         # Step 4：輸入密碼
#         {"type": "write",  "text": MY_PASSWORD},
#         # Step 5：按下登入按鈕
#         {"type": "click",  "selector": "input[type='submit']"},
#         # Step 6：等待登入完成後跳轉
#         {"type": "wait",   "milliseconds": 4000},
#         # Step 7：跳轉到目標討論版
#         {"type": "click",  "selector": "a[href*='bsn=76999']"},
#         {"type": "wait",   "milliseconds": 2000},
#     ]
# )
# print("=== 登入後討論版內容（前 500 字）===")
# print((result_login.markdown or "")[:500])

print("（登入範例已暫時註解，填入真實巴哈帳號後再執行）")
print("登入頁：https://user.gamer.com.tw/login.php")

---
## UNIT 4｜資料清洗、情緒統計與 Google Sheets 串接

### 4-A｜資料清洗 Pipeline

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = re.sub(r"<[^>]+>",  "",  text)
    text = re.sub(r"http\S+",  "",  text)
    text = re.sub(r"\s+",      " ", text)
    return text.strip()

# 套用到多輪爬取的結果
df_all["clean_title"] = df_all["title"].apply(clean_text)
df_clean = (df_all[df_all["clean_title"].str.len() > 5]
              .drop_duplicates(subset=["clean_title"])
              .reset_index(drop=True)
              .copy())
df_clean["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M")
df_clean["source"]     = "Gamer_巴哈_76999"

print(f"清洗前：{len(df_all)} 筆  →  清洗後：{len(df_clean)} 筆")
print(df_clean[["clean_title","source","scraped_at"]].head(8).to_string(index=False))

### 4-B｜jieba 批次情緒標記（套用至多輪爬取資料）

In [ ]:
df_clean["sentiment"] = df_clean["clean_title"].apply(rule_sentiment)

print("=== 情緒分布 ===")
print(df_clean["sentiment"].value_counts())
print()
print("=== 前 10 筆結果 ===")
print(df_clean[["clean_title","sentiment"]].head(10).to_string(index=False))

### 4-C｜視覺化情緒分布（matplotlib）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc("font", family="sans-serif")  # Colab 預設字型

labels  = df_clean["sentiment"].value_counts().index.tolist()
sizes   = df_clean["sentiment"].value_counts().values.tolist()
colors  = ["#27AE60", "#BDC3C7", "#E74C3C"][:len(labels)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 圓餅圖
axes[0].pie(sizes, labels=labels, colors=colors,
            autopct="%1.1f%%", startangle=90)
axes[0].set_title("情緒分布圓餅圖")

# 長條圖
axes[1].bar(labels, sizes, color=colors)
axes[1].set_title("情緒分布長條圖")
axes[1].set_ylabel("篇數")
for i, v in enumerate(sizes):
    axes[1].text(i, v + 0.3, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("sentiment_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅  圖表已儲存為 sentiment_chart.png")

### 4-D｜匯出 CSV / 寫入 Google Sheets

In [ ]:
# ── 方法 A：匯出 CSV 並下載 ──────────────────────────────────────────────────
cols = ["clean_title","url","sentiment","source","scraped_at"]
df_clean[cols].to_csv("gamer_sentiment_result.csv", index=False, encoding="utf-8-sig")
print("✅  已匯出 gamer_sentiment_result.csv")

from google.colab import files
files.download("gamer_sentiment_result.csv")

In [ ]:
# ── 方法 B：寫入 Google Sheets（需上傳服務帳號 JSON 金鑰）──────────────────
# from google.colab import files
# files.upload()

# import gspread
# from google.oauth2.service_account import Credentials
# creds = Credentials.from_service_account_file(
#     "your-key.json",
#     scopes=["https://www.googleapis.com/auth/spreadsheets",
#             "https://www.googleapis.com/auth/drive"]
# )
# gc = gspread.authorize(creds)
# ws = gc.open("社群輿情監控").sheet1
# df_export = df_clean[["clean_title","url","sentiment","source","scraped_at"]]
# ws.clear()
# ws.update([df_export.columns.tolist()] + df_export.values.tolist())
# print(f"✅  已寫入 {len(df_export)} 筆")

print("（Google Sheets 範例已暫時註解，請依前置步驟設定後啟用）")

---
## BONUS｜輿情摘要報告（純 Python，零成本）

In [ ]:
def generate_report(df) -> str:
    total = len(df)
    if total == 0: return "無資料"
    pos = (df["sentiment"]=="正向").sum()
    neu = (df["sentiment"]=="中立").sum()
    neg = (df["sentiment"]=="負向").sum()

    pos_ex = df[df["sentiment"]=="正向"]["clean_title"].head(3).tolist()
    neg_ex = df[df["sentiment"]=="負向"]["clean_title"].head(3).tolist()

    # 用 jieba 找出整體高頻關鍵字
    all_text = " ".join(df["clean_title"].tolist())
    top_kw   = [w for w, _ in jieba.analyse.extract_tags(all_text, topK=8, withWeight=True)]

    report = f"""
{'='*55}
📊  社群輿情摘要報告
{'='*55}
📅  分析時間：{datetime.now().strftime('%Y-%m-%d %H:%M')}
📌  資料來源：巴哈姆特寒霜啟示錄討論區
{'─'*55}
📈  情緒分布
    正向 😊  {pos:3d} 篇  ({pos/total*100:.1f}%)  {'█'*int(pos/total*30)}
    中立 😐  {neu:3d} 篇  ({neu/total*100:.1f}%)  {'█'*int(neu/total*30)}
    負向 😠  {neg:3d} 篇  ({neg/total*100:.1f}%)  {'█'*int(neg/total*30)}
{'─'*55}
🔑  熱門關鍵字：{'、'.join(top_kw)}
{'─'*55}
✅  正向貼文代表：
{chr(10).join(f'   · {t}' for t in pos_ex)}
❌  負向貼文代表：
{chr(10).join(f'   · {t}' for t in neg_ex)}
{'='*55}"""
    return report

print(generate_report(df_clean))

---
## 🐛 常見問題排查

| 問題 | 原因 | 解法 |
|------|------|------|
| `'Firecrawl' has no attribute 'scrape_url'` | 舊版 import | 用 `from firecrawl.v1 import V1FirecrawlApp` |
| `result.markdown` 回傳空白 | JS 渲染未完成 | 增加 `wait_for=5000` |
| jieba 斷詞結果不準 | 預設詞典無新詞 | 用 `jieba.add_word("新詞")` 自訂 |
| 情緒分類全是中立 | 關鍵字字典未命中 | 在 `positive_words` / `negative_words` 補充領域詞彙 |
| BeautifulSoup 找不到貼文 | CSS class 更新 | 改用 `soup.find_all("article")` |
| matplotlib 中文亂碼 | 字型問題 | 執行 `!apt-get install fonts-noto-cjk -q` 後重啟 Runtime |


In [ ]:
print("✅  Notebook 載入完畢！從第 0 步依序執行即可 🎉")